# Unpacker.py
Unpickles the CIFAR-10 Dataset and recompile it into DataLoader objects of an adequate size to run on a personal computer.

# Assignment 3
### CSC 537: Deep Learning

**Author:** Xander Palermo — ajp2s@missouristate.edu
**Professor:** Mukulika Ghosh
**Date:** 3 April 2026

---

### Imports

Libraries

In [18]:
import os
import pickle
from collections import defaultdict

import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2

Functions

In [19]:
# given unpacking method <https://www.cs.toronto.edu/~kriz/cifar.html>
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

Global Variables

In [20]:
metadata = unpickle('cifar-10-batches-py/batches.meta')
classes = [c.decode('utf8') for c in metadata[b'label_names']]

TRAINING_SIZE = 15000
VALIDATION_SIZE = 10000
TESTING_SIZE = 10000

### Unpack Data
Reads binary files provided and restructures the dataset as images in local directory

In [21]:
def unpack() -> None:


    # Metadata
    ## Check classification and verify directories

    for c in classes:
        os.makedirs('./dataset', exist_ok=True)
        os.makedirs('./dataset/complete', exist_ok=True)
        os.makedirs('./dataset/complete/training', exist_ok=True)
        os.makedirs('./dataset/complete/training/' + c, exist_ok=True)
        os.makedirs('./dataset/complete/testing', exist_ok=True)
        os.makedirs('./dataset/complete/testing/' + c, exist_ok=True)

    ## Images
    ## Unpickle each binary file and save the content as images (according to classification)

    for index in range(1, 6):
        f_name = f'cifar-10-batches-py/data_batch_{index}'
        d = unpickle(f_name)


        images = d[b'data']
        labels = d[b'labels']
        for i, (image, label) in enumerate(zip(images, labels)):
            image = image.reshape(3,32,32).transpose((1,2,0))
            label = classes[label]
            Image.fromarray(image).save(f'./dataset/complete/training/{label}/{i}.png')

        print(f"Completed batch {index}")

    f_name = f'cifar-10-batches-py/test_batch'
    d = unpickle(f_name)

    ## Repeat process for testing dataset
    images = d[b'data']
    labels = d[b'labels']
    for i, (image, label) in enumerate(zip(images, labels)):
        image = image.reshape(3,32,32).transpose((1,2,0))
        label = classes[label]
        Image.fromarray(image).save(f'./dataset/complete/testing/{label}/{i}.png')

    print(f"Completed testing set")

unpack()
print("Complete")

C:\Users\apale\AppData\Local\Temp\ipykernel_47580\2388273364.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict = pickle.load(fo, encoding='bytes')


Completed batch 1
Completed batch 2
Completed batch 3
Completed batch 4
Completed batch 5
Completed testing set
Complete


### Repack Data

Calculate mean and std for data normalization

In [22]:
mean = torch.zeros(3)
std = torch.zeros(3)
n_samples = 0

raw_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

raw_dataset = datasets.ImageFolder('./dataset/complete/training', transform=raw_transform)
loader = DataLoader(raw_dataset, batch_size=64, shuffle=False)

## Accumulate running totals
for images, _ in loader:
    batch_size = images.size(0)

    images = images.view(batch_size, 3, -1)

    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    n_samples += batch_size

## Avg.
mean /= n_samples
std /= n_samples

print(f'{mean=}\n{std=}')

mean=tensor([0.4910, 0.4819, 0.4465])
std=tensor([0.2021, 0.1993, 0.2009])


Assemble transformation stacks and apply to full dataset

In [23]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.RandomCrop(size=32, padding=4),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean.tolist(), std=std.tolist()),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean.tolist(), std=std.tolist()),
])

# Apply
full_training = datasets.ImageFolder('./dataset/complete/training', transform=train_transform)
full_testing  = datasets.ImageFolder('./dataset/complete/testing',  transform=test_transform)

condense dataset to smaller sections; section out validation subset

In [24]:
class_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_training.samples):
    class_indices[label].append(idx)

train_indices = []
val_indices   = []

num_samples_train =     TRAINING_SIZE   // len(classes)
num_samples_validate =  VALIDATION_SIZE // len(classes)
num_samples_test =      TESTING_SIZE    // len(classes)

for indices in class_indices.values():
    train_indices.extend(indices[:num_samples_train])
    val_indices.extend(indices[num_samples_train:num_samples_train + num_samples_validate])

training   = Subset(full_training, train_indices)
validation = Subset(full_training, val_indices)
testing    = Subset(full_testing,  range(TESTING_SIZE))

print(f'{len(training)=}\n{len(validation)=}\n{len(testing)=}')

len(training)=15000
len(validation)=10000
len(testing)=10000


Create DataLoader objects to give to SimpleCNN

In [25]:
training_loader = DataLoader(
    training,
    batch_size=64,
    shuffle=True,
    num_workers=4,
)

validate_loader = DataLoader(
    validation,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

testing_loader = DataLoader(
    testing,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

Pickle and save DataLoader objects

In [26]:
with open("./dataset/training.pkl", "wb") as f:
    pickle.dump(training_loader, f)

with open("./dataset/validation.pkl", "wb") as f:
    pickle.dump(validate_loader, f)

with open("./dataset/testing.pkl", "wb") as f:
    pickle.dump(testing_loader, f)